# Control de temperatura de la plataforma ThermoLab

En esta práctica vamos a aplicar los conceptos teóricos aprendidos para controlar la temperatura de un sistema térmico. Usaremos los conceptos de identificación de modelos, control PI, control PID y límites impuestos por la saturación y el retardo para diseñar un sistema de control realimentado.


## Carga del paquete ThermoLab y definición del sistema

Así cargamos el paquete ThermoLab y definimos la planta con la cual estamos trabajando. Si sale algún error corra la celda nuevamente.

In [ ]:
# cargar el paquete
using ThermoLab

# definicion del sistema
planta = ThermalSystem();

## Identificación del sistema - Modelo estático

Ahora obtendremos un modelo estático del sistema, fijando diferentes temperaturas de referencia y midiendo la potencia de estado estacionario. Esto se hace mediante la siguiente función (tarda unos minutos):

In [ ]:
get_static_model(planta);

### Preguntas orientadoras sobre el modelo estático

Discuta inmediatamente estas preguntas con su compañero e incluya una respuesta breve en este reporte.

+  ¿Qué significado físico tiene el término independiente $b$ del modelo estático?
+  ¿Cuál es la máxima temperatura que se puede alcanzar con el 100 % de la potencia? ¿Es seguro operar allí?

## Identificación del sistema - Modelo dinámico

A continuación vamos a obtener un modelo dinámico de la planta usando como entrada una onda binaria pseudoaleatoria
en el punto de operación en que la temperatura es $50\,^oC$. Recordemos que la función de transferencia simplificada tiene la forma:

$$G(s)=\frac{Y(s)}{U(s)}=\frac{b}{s+a}$$

In [ ]:
G, L = get_model_prbs(planta, yop=50);

## Control de temperatura con un controlador PI - Sintonía algebraica

A partir del modelo obtenido vamos a realizar un control PI de dos grados de libertad de acuerdo con la teoría vista. Tenemos las siguientes especificaciones para ese sistema de control:


| Controlador | $t_{ee}$ (tiempo de establecimiento) | $t_r$ (tiempo de subida) | $SP$ (sobrepico) |
|-------------|-----------|-------|------|
| PI-TEMP-1        | $\leq\, 60 s$     | $\leq\, 25 s$ | $\approx 5\%$  |

Además sabemos que el retardo del sistema de control en lazo cerrado es $0.8\,s$, debido al muestreo digital, y que la potencia está limitada entre $0$ y $100\,\%$.

1. Calcule el valor de $\zeta$ necesario para lograr el sobrepico deseado.
2. Encuentre el rango de valores de $\omega_n$ (inferior y superior) que puede colocarse para constantes no negativas, cumplir  las especificaciones y el retardo inherente al muestreo digital ($\omega_n\leq 0.5/L$, siendo $L$ el retardo).

3. Realice iteraciones experimentales progresivas con el fin de ajustar el valor de $\omega_n$ hacia el valor mayor posible, manteniendo las especificaciones. Es posible que tenga que ajustar experimentalmente $\zeta$ después del ajuste de $\omega_n$.

4. Use un valor de $\omega_n$ muy por encima del valor que satisface las especificaciones y observe la saturación de la potencia. Tenga cuidado de ajustar el valor de $t_1$ en la función ``step_closed`` [(vea la ayuda)](https://nebisman.github.io/ThermoLab.jl/dev/funciones/control/step_closed/) para que la temperatura llegue al estado estacionario.  Discuta con su compañero lo que pasa e incluya sus reflexiones sobre los comportamientos observados.

In [ ]:
a =
b =
ωn =
ζ =

kp =
ki =

# calculo de las constantes, incluya aqui el calculo de las constantes del PI, kp, ki
# con base en los valores de ωn y ζ, y los valores de a y b

# función de lazo cerrado teorica
s = tf("s")
T = ωn^2/(s^2 + 2*ζ*ωn*s + ωn^2)

# aqui programo el PI
set_pid(planta,  kp=kp, ki=ki, kd=0, beta=0)

# aqui ejecuto el experimento de escalón de temperatura de 40 a 50 grados.
resultado = step_closed(planta, r0 = 40, r1 = 50,  t0 = 60, t1 = 12/(ζ*ωn));

# aqui visualizo las metricas y comparo con el desempeño deseado de la función de lazo cerrado T
stepinfo(resultado, T)

## Control de temperatura con un controlador PID - modelo de dos polos

Ahora vamos a obtener, con los mismos datos PRBS, un modelo de dos polos de la planta:

$$G_2(s)=\dfrac{Y(s)}{U(s)}=\dfrac{b}{(s+a_1)(s+a_2)}$$

In [ ]:
G2, L = get_model_prbs(planta; numpar=3, usefile=true)

### Pregunta orientadora

+  Compare el ajuste (FIT) de los modelos de uno y de dos polos. ¿Qué fenómeno físico explica el segundo polo?

## Diseño de un controlador PID

A partir del modelo de dos polos,  vamos a diseñar un control PID de dos grados de libertad. Tenemos las siguientes especificaciones:


| Controlador | $t_{ee}$ (tiempo de establecimiento) | $t_r$ (tiempo de subida) | $SP$ (sobrepico) |
|-------------|-----------|-------|------|
| PID-TEMP-1        | $\leq\,30 s$     | $\leq\,10 s$ | $\leq 5\%$  |


Adicionalmente, la restricción de energía impone que:

$$0 \leq u(t)\leq 100\,\%$$

La referencia de temperatura a lo sumo va a cambiar en $10$ grados.


1. Encuentre la región de diseño que satisface las especificaciones.

2. Encuentre el rango de valores de $\omega_n$ (inferior y superior) que puede colocarse de acuerdo con las especificaciones y el retardo.

3. Use las fórmulas para $k_p$, $k_i$ y $k_d$ que dedujo en clase, junto con el modelo $G_2$. Puede usar para  $m$ un valor entre 2 y 3.

4. Realice iteraciones experimentales progresivas ajustando los polos de lazo cerrado, de manera que se satisfagan las especificaciones y no se sature la potencia cuando se aplica un escalón de $40$ a $50\,^oC$ con la función ``step_closed``.

In [ ]:
## Calculo de las constantes del PID para el sistema de segundo orden


ωn =
ζ =
m =

# funcion de lazo cerrado teorica
T2  = m*ωn^3/((s+m*ωn)*(s^2 + 2*ζ*ωn*s +ωn^2))

# complete las lineas de codigo para calcular las constantes del PID

kp =
ki =
kd =

set_pid(planta,  kp=kp, ki=ki, kd=kd, N=10, beta=0)

# complete las lineas de código para ejecutar el experimento de escalón de temperatura de 40 a 50 grados
# y visualizar las métricas y compararlas con el desempeño deseado de la función de lazo cerrado T2 con stepinfo

### Reflexiones

Incluya sus reflexiones sobre esta práctica.